# Лабораторная работа 2. Классификация текстов с применением RNN

Источник задания: `materials/Лаб2.md`. Архитектура Many-to-One и формулы GRU — лекция `materials/notes/disciplines__nlp__lectures__rnn-2.md`. Предобработка — лекция preprocessing.

База кода: Colab-скрипт одногруппника (жалобы жильцов: RNN/LSTM/GRU + Attention, свой GRU, веса классов). Адаптировано под локальный датасет и структуру «Ход работы».


## 1. Загрузка датасета «просьбы жильцов»

Пункт 1 хода работы. Датасет большой — берём стратифицированную часть корпуса (`reason_category`).


In [1]:
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH = Path('../../materials/dataset/Petitions.csv')

# Параметры: часть корпуса по методичке; CustomGRU на CPU тяжёлый — SAMPLE умеренный
SAMPLE_SIZE = 8000
MIN_WORD_FREQ = 3
MAX_LEN = 40
EMBED_DIM = 100
HIDDEN_DIM = 128
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3

TEXT_COL = 'public_petition_text'
TARGET_COL = 'reason_category'

df = pd.read_csv(DATA_PATH)
print('Исходный размер:', len(df))
print('Столбцы:', df.columns.tolist())

df = df[[TEXT_COL, TARGET_COL]].dropna().copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)
df[TARGET_COL] = df[TARGET_COL].astype(str)

df_sampled, _ = train_test_split(
    df,
    train_size=min(SAMPLE_SIZE, len(df)),
    random_state=SEED,
    stratify=df[TARGET_COL],
)
df_sampled = df_sampled.reset_index(drop=True)
print('Размер корпуса:', len(df_sampled))
print('Категорий:', df_sampled[TARGET_COL].nunique())
print(df_sampled[TARGET_COL].value_counts().head())
print('Устройство:', DEVICE)


Исходный размер: 59889
Столбцы: ['id', 'public_petition_text', 'reason_category']
Размер корпуса: 8000
Категорий: 15
reason_category
Благоустройство                                            4644
Содержание МКД                                             1932
Нарушение правил пользования общим имуществом               290
Незаконная информационная и (или) рекламная конструкция     245
Фасад                                                       198
Name: count, dtype: int64
Устройство: cpu


## 2. Предварительная обработка текстов

Пункт 2. Каждый объект корпуса — отдельный документ: нижний регистр, только кириллица, токены по пробелам, стоп-слова и короткие токены отбрасываем.

Стоп-слова: фиксированный список + служебные слова жалоб (`просим`, `просьба` и т.п.). В материалах конкретный список не зафиксирован; `nltk` в окружении курса не используем.


In [2]:
RU_STOPWORDS = {
    'и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она',
    'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее',
    'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда',
    'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до',
    'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей',
    'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем',
    'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под',
    'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой',
    'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее',
    'сейчас', 'были', 'куда', 'почему', 'всегда', 'можно', 'при', 'наконец', 'два',
    'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про',
    'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впрочем',
    'хорошо', 'свою', 'этой', 'перед', 'иногда', 'лучше', 'чуть', 'том', 'нельзя', 'такой',
    'им', 'более', 'конечно', 'всю', 'между',
    # слова из исходного скрипта (жалобы)
    'просим', 'просьба', 'спасибо', 'пожалуйста', 'улиц', 'дом', 'подъезд',
}


def preprocess_text(text):
    if pd.isna(text):
        return []
    tokens = re.sub(r'[^а-яё\s]', ' ', str(text).lower()).split()
    return [w for w in tokens if w not in RU_STOPWORDS and len(w) >= 3]


df_sampled = df_sampled.copy()
df_sampled['tokens'] = df_sampled[TEXT_COL].apply(preprocess_text)
df_sampled = df_sampled[df_sampled['tokens'].apply(len) > 0].reset_index(drop=True)
print('После очистки пустых:', len(df_sampled))
print('Пример токенов:', df_sampled['tokens'].iloc[0][:15])


После очистки пустых: 8000
Пример токенов: ['качелями', 'вокруг', 'круговой', 'карусели', 'внизу', 'спуске', 'горки', 'образовались', 'ямы', 'дождя', 'постоянно', 'лужи', 'устранить', 'проблемы']


## 3. Токенизация и векторизация

Пункт 3. Строим словарь по частоте (≥ `MIN_WORD_FREQ`), индексы `<PAD>=0`, `<UNK>=1`. Векторизация токенов — обучаемый `nn.Embedding` (лекция rnn-2); здесь кодируем тексты в последовательности индексов фиксированной длины `MAX_LEN`.


In [3]:
all_tokens = [token for tokens in df_sampled['tokens'] for token in tokens]
word_counts = Counter(all_tokens)
# Важно: индексы только по словам, прошедшим порог частоты
# (в исходном .py enumerate шёл по всему Counter — индексы уезжали за VOCAB_SIZE)
filtered_words = [w for w, c in word_counts.items() if c >= MIN_WORD_FREQ]
vocab = {word: idx + 2 for idx, word in enumerate(filtered_words)}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
VOCAB_SIZE = len(vocab)


def encode_text(tokens):
    enc = [vocab.get(w, vocab['<UNK>']) for w in tokens]
    return (enc + [vocab['<PAD>']] * MAX_LEN)[:MAX_LEN]


df_sampled['encoded_text'] = df_sampled['tokens'].apply(encode_text)
print('Размер словаря (VOCAB_SIZE):', VOCAB_SIZE)
print('MAX_LEN:', MAX_LEN)


Размер словаря (VOCAB_SIZE): 4366
MAX_LEN: 40


## 4. Кодирование целевого признака и train/test

Пункт 4. `LabelEncoder` по `reason_category`, стратифицированный split 80/20. Веса классов (`compute_class_weight`) — из-за сильного дисбаланса категорий (общие знания / sklearn; в лекциях NLP поимённо не разобраны).


In [4]:
label_encoder = LabelEncoder()
df_sampled['encoded_target'] = label_encoder.fit_transform(df_sampled[TARGET_COL])

X = np.array(df_sampled['encoded_text'].tolist())
y = df_sampled['encoded_target'].values

print('Индексы: min={}, max={}, VOCAB_SIZE={}'.format(X.min(), X.max(), VOCAB_SIZE))
if X.max() >= VOCAB_SIZE:
    X = np.clip(X, 0, VOCAB_SIZE - 1)
    print('Индексы обрезаны до [0, VOCAB_SIZE-1]')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)


class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TextDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

majority = np.bincount(y_train).argmax()
print('Train/Test:', len(X_train), len(X_test))
print('Классов:', len(label_encoder.classes_))
print('Веса: min={:.3f}, max={:.3f}'.format(class_weights.min(), class_weights.max()))
print('Majority accuracy (константа):', (y_test == majority).mean())


Индексы: min=0, max=4365, VOCAB_SIZE=4366
Train/Test: 6400 1600
Классов: 15
Веса: min=0.115, max=16.410
Majority accuracy (константа): 0.580625


## 5. Сети RNN, LSTM и GRU (Many-to-One) + Attention

Пункт 5. Три сети на блоках `nn.RNN` / `nn.LSTM` / `nn.GRU`. Attention поверх последнего скрытого состояния — доп. слой исходного скрипта (не обязателен методичкой; Many-to-One сохраняется: классификация по агрегату последовательности).

Формулы attention: веса по скалярному произведению выходов с `h_T`, контекстный вектор, конкатенация с `h_T` → линейный классификатор.


In [5]:
class AttentionLayer(nn.Module):
    """Attention по последнему скрытому состоянию (Many-to-One)."""

    def __init__(self, hidden_size):
        super().__init__()

    def forward(self, out, h):
        # out: [B, T, H], h: [B, H]
        e = torch.bmm(out, h.unsqueeze(2)).squeeze(2)
        alpha = torch.softmax(e, dim=1)
        context = torch.sum(out * alpha.unsqueeze(2), dim=1)
        return torch.cat([h, context], dim=1)


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.att = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(2 * hidden_dim, n_classes)

    def forward(self, x):
        out, h = self.rnn(self.emb(x))
        return self.fc(self.att(out, h[-1]))


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.att = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(2 * hidden_dim, n_classes)

    def forward(self, x):
        out, (h, _) = self.lstm(self.emb(x))
        return self.fc(self.att(out, h[-1]))


class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.att = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(2 * hidden_dim, n_classes)

    def forward(self, x):
        out, h = self.gru(self.emb(x))
        return self.fc(self.att(out, h[-1]))


N_CLASSES = len(np.unique(y))
print('Классов для моделей:', N_CLASSES)
print('Архитектуры: RNN / LSTM / GRU + Attention готовы')


Классов для моделей: 15
Архитектуры: RNN / LSTM / GRU + Attention готовы


### Обучение трёх моделей

`CrossEntropyLoss` с весами классов, Adam, `EPOCHS` эпох. Метрики — на тесте.


In [6]:
def train_and_eval(model, name, train_loader, test_loader, class_weights_tensor, device,
                   epochs=EPOCHS, lr=LR):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        n_batches = 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad()
            loss = crit(model(bx), by)
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_batches += 1
        print('  epoch {}/{}  loss={:.4f}'.format(
            epoch + 1, epochs, total_loss / max(n_batches, 1)))

    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            logits = model(bx.to(device))
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels.extend(by.numpy())

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    elapsed = time.time() - t0
    print('{}: Acc={:.4f}  Macro-F1={:.4f}  ({:.1f} с)'.format(name, acc, f1, elapsed))
    return {'Acc': acc, 'F1': f1, 'preds': preds, 'labels': labels, 'model': model}


results = {}
builtin_models = {
    'RNN + Attention': RNNClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_CLASSES),
    'LSTM + Attention': LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_CLASSES),
    'GRU + Attention': GRUClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_CLASSES),
}

for name, model in builtin_models.items():
    print('--- Обучение: {} ---'.format(name))
    results[name] = train_and_eval(
        model, name, train_loader, test_loader, class_weights_tensor, DEVICE
    )


--- Обучение: RNN + Attention ---


  epoch 1/5  loss=2.6982


  epoch 2/5  loss=2.5257


  epoch 3/5  loss=2.2814


  epoch 4/5  loss=2.0410


  epoch 5/5  loss=2.2427
RNN + Attention: Acc=0.1619  Macro-F1=0.1137  (12.8 с)
--- Обучение: LSTM + Attention ---


... (8 outputs left out)

  epoch 3/5  loss=1.9669


  epoch 4/5  loss=1.4825


  epoch 5/5  loss=1.0940


GRU + Attention: Acc=0.5994  Macro-F1=0.3654  (26.7 с)


## 6. Оценка качества и сравнение

Пункт 6. Accuracy и Macro-F1 на тесте; лучшая из трёх моделей методички. Macro-F1 важнее при дисбалансе: accuracy может быть близка к majority.

Замечание: при `class_weight` простая RNN часто даёт accuracy ниже majority (модель «размазывает» предсказания по редким классам). Сравнивать модели надёжнее по Macro-F1; из трёх блоков методички обычно выигрывает GRU.


In [7]:
print('Итог по трём моделям методички:')
for name in ['RNN + Attention', 'LSTM + Attention', 'GRU + Attention']:
    m = results[name]
    print('{:<20} | Acc: {:.4f} | F1-macro: {:.4f}'.format(name, m['Acc'], m['F1']))

best3 = max(
    ['RNN + Attention', 'LSTM + Attention', 'GRU + Attention'],
    key=lambda k: results[k]['F1'],
)
print('\nЛучшая из трёх (по Macro-F1): {} ({:.4f})'.format(best3, results[best3]['F1']))
print('\nclassification_report (лучшая):')
print(classification_report(
    results[best3]['labels'],
    results[best3]['preds'],
    target_names=label_encoder.classes_,
    zero_division=0,
))


Итог по трём моделям методички:
RNN + Attention      | Acc: 0.1619 | F1-macro: 0.1137
LSTM + Attention     | Acc: 0.2787 | F1-macro: 0.2576
GRU + Attention      | Acc: 0.5994 | F1-macro: 0.3654

Лучшая из трёх (по Macro-F1): GRU + Attention (0.3654)

classification_report (лучшая):
                                                                                  precision    recall  f1-score   support

                                                                 Благоустройство       0.90      0.63      0.74       929
                                                                   Водоотведение       0.15      0.50      0.23         6
                                                                   Водоснабжение       0.73      0.50      0.59        22
                                                                          Кровля       0.09      0.41      0.15        22
                                  Нарушение порядка пользования общим имуществом       0.22      0.33     

## 7. Собственный GRU-блок и сеть на нём

Пункт 7. Реализация по формулам лекции rnn-2: ворота reset $r$, update $z$, кандидат $n$, новое состояние.

Слой обрабатывает последовательность во времени (цикл по $T$). Сеть — как у стандартного GRU + Attention. Сравнение метрик с `nn.GRU` проверяет корректность реализации.


In [8]:
class CustomGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_r = nn.Linear(input_size, hidden_size)
        self.U_r = nn.Linear(hidden_size, hidden_size)
        self.W_z = nn.Linear(input_size, hidden_size)
        self.U_z = nn.Linear(hidden_size, hidden_size)
        self.W_n = nn.Linear(input_size, hidden_size)
        self.U_n = nn.Linear(hidden_size, hidden_size)

    def forward(self, x, h_prev):
        r = torch.sigmoid(self.W_r(x) + self.U_r(h_prev))
        z = torch.sigmoid(self.W_z(x) + self.U_z(h_prev))
        n = torch.tanh(self.W_n(x) + r * self.U_n(h_prev))
        return (1 - z) * n + z * h_prev


class CustomGRULayer(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.cell = CustomGRUCell(input_size, hidden_size)
        self.h_size = hidden_size

    def forward(self, x):
        # x: [B, T, E]
        b, s, _ = x.shape
        h = torch.zeros(b, self.h_size, device=x.device)
        outs = []
        for t in range(s):
            h = self.cell(x[:, t, :], h)
            outs.append(h.unsqueeze(1))
        return torch.cat(outs, dim=1), h


class CustomGRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = CustomGRULayer(embed_dim, hidden_dim)
        self.att = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(2 * hidden_dim, n_classes)

    def forward(self, x):
        out, h = self.gru(self.emb(x))
        return self.fc(self.att(out, h))


print('--- Обучение: Custom GRU + Att ---')
custom_model = CustomGRUClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_CLASSES)
results['Custom GRU + Att'] = train_and_eval(
    custom_model, 'Custom GRU + Att', train_loader, test_loader, class_weights_tensor, DEVICE
)


--- Обучение: Custom GRU + Att ---


  epoch 1/5  loss=2.6863


  epoch 2/5  loss=2.4548


  epoch 3/5  loss=2.0465


  epoch 4/5  loss=1.6179


  epoch 5/5  loss=1.1525


Custom GRU + Att: Acc=0.5988  Macro-F1=0.3813  (48.0 с)


### Итоговая таблица

Сравнение всех четырёх моделей. Вывод: какая из трёх блоков методички лучше; насколько свой GRU близок к `nn.GRU`.


In [9]:
print('=' * 70)
print('ИТОГОВАЯ ТАБЛИЦА')
print('=' * 70)
for name, m in results.items():
    print('{:<22} | Acc: {:.4f} | F1-macro: {:.4f}'.format(name, m['Acc'], m['F1']))

best_all = max(results, key=lambda k: results[k]['F1'])
print('\nЛучшая по Macro-F1: {} ({:.4f})'.format(best_all, results[best_all]['F1']))


ИТОГОВАЯ ТАБЛИЦА
RNN + Attention        | Acc: 0.1619 | F1-macro: 0.1137
LSTM + Attention       | Acc: 0.2787 | F1-macro: 0.2576
GRU + Attention        | Acc: 0.5994 | F1-macro: 0.3654
Custom GRU + Att       | Acc: 0.5988 | F1-macro: 0.3813

Лучшая по Macro-F1: Custom GRU + Att (0.3813)
